# Syrin — Complete Guide

**The single source of truth for the entire Syrin library.**

Run every cell in order. No API key required — all examples use `Model.mock()` by default.
Set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` in the Setup cell to use real models.

**What's covered:**

| # | Topic | What you'll learn |
|---|-------|-------------------|
| 1 | Your First Agent | Class-based agents, `run()`, `Response` object |
| 2 | Budget & Cost Control | Hard limits, `ExceedPolicy`, thresholds, rate limits |
| 3 | Tools | `@tool`, tool calls, TOON format |
| 4 | Memory | 4 memory types, `remember()`, `recall()` |
| 5 | Structured Output | `@structured`, typed responses |
| 6 | Hooks & Observability | Lifecycle events, cost tracking, debug mode |
| 7 | Context | Window management, snapshots, compaction |
| 8 | Guardrails | Content filtering, `GuardrailChain` |
| 9 | Checkpoints | Save/load agent state |
| 10 | **Swarm: Parallel** | All agents run simultaneously, results merged |
| 11 | **Swarm: Orchestrator** | LLM-driven dynamic task delegation |
| 12 | **Swarm: Consensus** | Multi-agent voting for high-stakes decisions |
| 13 | **Swarm: Reflection** | Writer + critic loop for quality improvement |
| 14 | **Swarm: Workflow** | Sequential, parallel, branch, and dynamic steps |
| 15 | **Budget Delegation** | `self.spawn()`, `self.spawn_many()`, pool slicing |
| 16 | **MemoryBus** | Cross-agent shared memory with access control |
| 17 | **A2A Messaging** | Typed agent-to-agent direct, broadcast, topic |
| 18 | **Runtime Budget Control** | `topup_budget()`, `reallocate_budget()` mid-run |
| 19 | **AgentRouter** | LLM-driven dynamic routing across agent pool |
| 20 | **Hierarchical Swarms** | Org-chart agent delegation (CEO → CTO → workers) |
| 21 | AgentRegistry | Discover and control running agents |
| 22 | Serving | HTTP endpoint, web playground |
| 23 | Real-World: Research Pipeline | Full end-to-end multi-agent example |

---
## Setup

In [ ]:
# Install Syrin and dependencies
# !pip install syrin openai anthropic pydantic tiktoken nest_asyncio

In [ ]:
# ── API keys (optional) ──────────────────────────────────────────────────────
# Leave empty to use Model.mock() — all examples work without an API key.
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = ""

In [ ]:
import logging

logging.basicConfig(level=logging.ERROR)
for name in ("httpx", "httpcore", "asyncio"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

# nest_asyncio allows asyncio.run() inside Jupyter (multi-agent cells need this)
try:
    import nest_asyncio

    nest_asyncio.apply()
except ImportError:
    pass  # Not needed for kernels that support top-level await natively


def get_model():
    """Returns the best available model based on configured API keys."""
    from syrin import Model

    if OPENAI_API_KEY:
        return Model.OpenAI("gpt-4o-mini", api_key=OPENAI_API_KEY)
    if ANTHROPIC_API_KEY:
        return Model.Anthropic("claude-haiku-4-5", api_key=ANTHROPIC_API_KEY)
    return Model.mock()


active = "OpenAI" if OPENAI_API_KEY else "Anthropic" if ANTHROPIC_API_KEY else "Mock (no key)"
print(f"Active model: {active}")
print("All examples work with Mock. Real models produce better output.")

---
## 1. Your First Agent

Define an agent as a class. Set `model` and `system_prompt` as class attributes. Call `agent.run(input)` to get a `Response`.

In [ ]:
from syrin import Agent


class Assistant(Agent):
    model = get_model()
    system_prompt = "You are a helpful assistant. Be concise."


agent = Assistant()
result = agent.run("What is Python?")

print("Content:", result.content[:150])
print("Cost:   $", result.cost)
print("Tokens: ", result.tokens.total_tokens)
print("Model:  ", result.model)

In [ ]:
# Quick one-liner — syrin.run() without creating an agent
import syrin

result = syrin.run(
    "Explain machine learning in one sentence.", model=get_model(), system_prompt="Be brief."
)
print(result.content)

In [ ]:
# Agent inheritance — subclass to specialise
class BaseAgent(Agent):
    model = get_model()
    system_prompt = "You are helpful."


class ExpertAgent(BaseAgent):
    system_prompt = "You are a Python expert. Give detailed, precise answers."


print(ExpertAgent().run("What is a generator?").content[:200])

---
## 2. Budget & Cost Control

Every agent can have a hard cost limit. `ExceedPolicy` controls what happens when the limit is hit.

In [ ]:
from syrin import Budget
from syrin.enums import ExceedPolicy


class CostAwareAgent(Agent):
    model = get_model()
    system_prompt = "You are concise."
    budget = Budget(
        max_cost=0.10,
        exceed_policy=ExceedPolicy.WARN,  # STOP | WARN | IGNORE | SWITCH
    )


agent = CostAwareAgent()
result = agent.run("Explain neural networks.")
print(f"Cost: ${result.cost:.6f}")
print(f"Budget state: {agent.budget_state}")

In [ ]:
# Thresholds — fire a callback at percentage of budget used
from syrin.enums import ThresholdMetric
from syrin.threshold import BudgetThreshold, ThresholdContext

events = []


def on_80_pct(ctx: ThresholdContext) -> None:
    events.append(f"80% threshold hit — ${ctx.current_value:.6f} spent")


agent = Agent(
    model=get_model(),
    budget=Budget(
        max_cost=0.01,
        thresholds=[BudgetThreshold(at=80, action=on_80_pct, metric=ThresholdMetric.COST)],
        exceed_policy=ExceedPolicy.WARN,
    ),
)
agent.run("Hello!")
print("Threshold events:", events or ["(none fired — mock has near-zero cost)"])

In [ ]:
# Rate limits — cap spend per hour/day
from syrin import RateLimit

budget = Budget(
    max_cost=1.0,
    rate_limits=RateLimit(hour=5.0, day=50.0),
    exceed_policy=ExceedPolicy.WARN,
)
agent = Agent(model=get_model(), system_prompt="You are concise.", budget=budget)
result = agent.run("Hello")
print(f"Cost: ${result.cost:.6f} | Budget: {agent.budget_state}")

---
## 3. Tools

Decorate functions with `@tool`. The agent can call them during a run. Tools use TOON schema (40% fewer tokens than JSON).

In [ ]:
from syrin import tool


@tool
def calculate(a: float, b: float, operation: str = "add") -> str:
    """Perform arithmetic: add, subtract, multiply, divide."""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b else float("inf"),
    }
    return str(ops.get(operation, "unknown operation"))


@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get current weather for a city."""
    return f"{city}: 22°{unit[0].upper()}, partly cloudy"


class ToolAgent(Agent):
    model = get_model()
    system_prompt = "Use tools to answer questions. Don't guess numbers."
    tools = [calculate, get_weather]


agent = ToolAgent()
result = agent.run("What is 15 multiplied by 7?")
print("Answer:", result.content)
print("Tool calls:", len(result.tool_calls))

---
## 4. Memory

Agents with `memory=Memory()` can store and recall information across calls. Four types: `CORE`, `EPISODIC`, `SEMANTIC`, `PROCEDURAL`.

In [ ]:
from syrin import Memory, MemoryType

agent = Agent(
    model=get_model(),
    system_prompt="You are a personal assistant that remembers user preferences.",
    memory=Memory(),
)

# Store facts
agent.remember("The user's name is Alice.", memory_type=MemoryType.CORE, importance=1.0)
agent.remember("Alice prefers Python over JavaScript.", memory_type=MemoryType.SEMANTIC)
agent.remember("Alice asked about ML frameworks yesterday.", memory_type=MemoryType.EPISODIC)

# Recall relevant memories
memories = agent.recall("user preferences", top_k=3)
print("Recalled memories:")
for m in memories:
    print(f"  [{m.type}] {m.content}")

# Ask a question — agent uses memory as context
result = agent.run("What programming language does the user prefer?")
print("\nAnswer:", result.content[:100])

---
## 5. Structured Output

Use `@structured` to define typed output schemas. The agent validates and returns a typed object instead of raw text.

In [ ]:
from dataclasses import dataclass

from syrin import structured


@structured
@dataclass
class ProductAnalysis:
    name: str
    sentiment: str  # positive | negative | neutral
    score: float  # 0.0 – 1.0
    key_strength: str
    key_weakness: str


class AnalystAgent(Agent):
    model = get_model()
    system_prompt = "You analyse products and return structured assessments."
    output_type = ProductAnalysis


result = AnalystAgent().run(
    "Analyse: iPhone 16 — best camera on market, but expensive at $999 and no USB-C fast charging."
)

if result.output:
    p = result.output
    print(f"Name:     {p.name}")
    print(f"Sentiment: {p.sentiment} (score: {p.score})")
    print(f"Strength:  {p.key_strength}")
    print(f"Weakness:  {p.key_weakness}")
else:
    print("Raw output:", result.content[:100])

---
## 6. Hooks & Observability

Subscribe to lifecycle events with `agent.events.on(Hook.*, handler)`. Track cost, model calls, tool use, errors — everything.

In [ ]:
from syrin.enums import Hook

agent = Agent(model=get_model(), system_prompt="You are helpful.")

costs = []
agent.events.on(Hook.LLM_RESPONSE, lambda ctx: costs.append(ctx.get("cost", 0)))
agent.events.on(
    Hook.AGENT_COMPLETE, lambda ctx: print(f"  Done — total cost: ${ctx.get('cost', 0):.6f}")
)

agent.run("What is 2+2?")
agent.run("Name a planet.")
print(f"Calls: {len(costs)} | Total: ${sum(costs):.6f}")

In [ ]:
# debug=True enables full tracing to console (model, tokens, cost per call)
debug_agent = Agent(model=get_model(), system_prompt="You are helpful.", debug=True)
result = debug_agent.run("What is the capital of France?")
print("Answer:", result.content[:80])

---
## 7. Context

`Context` manages the agent's context window: token limits, compaction, and snapshots.

In [ ]:
from syrin.context import Context

agent = Agent(
    model=get_model(),
    system_prompt="You are a helpful assistant.",
    context=Context(max_tokens=8000),
)
agent.run("Tell me about Python.")

# Snapshot — inspect what was sent to the LLM
snap = agent.context.snapshot()
print(f"Context utilization: {snap.utilization_pct:.1f}%")
print(f"Context rot risk:    {snap.context_rot_risk}")
print(f"Why included:        {snap.why_included[:3]}")

---
## 8. Guardrails

Filter inputs and outputs. Block keywords, validate schemas, detect injections.

In [ ]:
from syrin.enums import GuardrailStage
from syrin.guardrails import ContentFilter, GuardrailChain

chain = GuardrailChain(
    [ContentFilter(blocked_words=["spam", "scam", "hack"], name="ContentPolicy")]
)

print("Clean input:   ", chain.check("Hello!", GuardrailStage.INPUT).passed)
print("Blocked input: ", chain.check("This is spam", GuardrailStage.INPUT).passed)

agent = Agent(model=get_model(), system_prompt="You are helpful.", guardrails=chain)
result = agent.run("Help me with Python")
print("Agent response:", result.content[:80])

---
## 9. Checkpoints

Save and restore agent state. Useful for long-running agents and crash recovery.

In [ ]:
agent = Agent(model=get_model(), system_prompt="You are a research assistant.", memory=Memory())
agent.remember("Researching: quantum computing market.", memory_type=MemoryType.CORE)
agent.run("Summarize the current state.")

# Save state
checkpoint_id = agent.save_checkpoint()
print("Saved checkpoint:", checkpoint_id)
print("All checkpoints:", agent.list_checkpoints())

# Restore in a new agent (simulating restart)
# agent2 = Agent.load_checkpoint(checkpoint_id)

---
## 10. Multi-Agent: Parallel Swarm

All agents run simultaneously on the same goal. Results are merged. Use when agents are **independent specialists** — fastest topology.

```
Goal ──→ [Agent A]  ──┐
      ──→ [Agent B]  ──┼──→ Merged result
      ──→ [Agent C]  ──┘
```

In [ ]:
import asyncio

from syrin.enums import FallbackStrategy, SwarmTopology
from syrin.swarm import Swarm, SwarmConfig


class MarketResearcher(Agent):
    model = get_model()
    system_prompt = "You analyse market size, growth rates, and dominant players. Be concise."


class CompetitiveAnalyst(Agent):
    model = get_model()
    system_prompt = "You surface competitive dynamics, pricing pressure, and threats. Be concise."


class ExecutiveBriefer(Agent):
    model = get_model()
    system_prompt = "You distil findings into a crisp 2-sentence recommendation. Be concise."


async def run_parallel_swarm():
    swarm = Swarm(
        agents=[MarketResearcher, CompetitiveAnalyst, ExecutiveBriefer],
        goal="AI developer tooling market — 2025 outlook",
        budget=Budget(max_cost=0.50),
        config=SwarmConfig(
            topology=SwarmTopology.PARALLEL,
            on_agent_failure=FallbackStrategy.SKIP_AND_CONTINUE,
        ),
    )
    result = await swarm.run()

    print("Merged output (first 300 chars):")
    print(result.content[:300])
    print("\nCost breakdown:")
    for name, cost in result.cost_breakdown.items():
        print(f"  {name}: ${cost:.6f}")

    # SwarmResult fields
    # result.content          — merged output from all agents
    # result.cost_breakdown   — {agent_name: cost}
    # result.agent_results    — list of Response from successful agents
    # result.partial_results  — list of Response when some agents failed
    # result.budget_report    — total_spent, per_agent summaries


asyncio.run(run_parallel_swarm())

---
## 11. Multi-Agent: Orchestrator

The **first agent** is the orchestrator. It reads the goal, decides which workers to call, and synthesises their outputs. Delegation is LLM-driven — it adapts per goal without you defining a fixed graph.

```
Goal ──→ [Orchestrator] ──→ [Worker A] ──┐
                       └──→ [Worker B] ──┼──→ Synthesised result
                       └──→ [Worker C] ──┘
```

In [ ]:
class ResearchDirector(Agent):
    """Orchestrator: coordinates the team and synthesises a final briefing."""

    model = get_model()
    system_prompt = (
        "You are a senior research director. Delegate sub-tasks to your team "
        "and synthesise their outputs into a structured briefing: "
        "Market Overview, Competitive Landscape, Strategic Recommendation."
    )


class MarketAnalyst(Agent):
    model = get_model()
    system_prompt = "You report market size, CAGR, and top-3 player revenue share. Be concise."


class StrategicAdvisor(Agent):
    model = get_model()
    system_prompt = "You translate findings into a 2-sentence investment verdict. Be concise."


async def run_orchestrator_swarm():
    swarm = Swarm(
        # First agent = orchestrator, rest = workers
        agents=[ResearchDirector, MarketAnalyst, StrategicAdvisor],
        goal="Generative AI infrastructure market — investment thesis 2025",
        budget=Budget(max_cost=0.50),
        config=SwarmConfig(topology=SwarmTopology.ORCHESTRATOR),
    )
    result = await swarm.run()

    print("Orchestrated result (first 300 chars):")
    print(result.content[:300])
    print(f"\nTotal cost: ${sum(result.cost_breakdown.values()):.6f}")


asyncio.run(run_orchestrator_swarm())

---
## 12. Multi-Agent: Consensus

Multiple agents evaluate the same question **independently** and vote. The result is the answer with the most agreement. Use for **high-stakes decisions** where a single model's answer carries unacceptable risk.

In [ ]:
from syrin.enums import ConsensusStrategy
from syrin.swarm import ConsensusConfig

CLAUSE = (
    "Clause 14.3: In no event shall either party be liable "
    "for consequential or indirect damages arising from use of the software."
)


class USCounsel(Agent):
    model = get_model()
    system_prompt = (
        "You are a US software contracts attorney. Assess the clause and end "
        "your response with exactly: ENFORCEABLE, UNENFORCEABLE, or JURISDICTION-DEPENDENT."
    )


class EUCounsel(Agent):
    model = get_model()
    system_prompt = (
        "You are an EU technology law specialist. Assess the clause and end "
        "your response with exactly: ENFORCEABLE, UNENFORCEABLE, or JURISDICTION-DEPENDENT."
    )


class CommonwealthCounsel(Agent):
    model = get_model()
    system_prompt = (
        "You are a Commonwealth jurisdiction solicitor. Assess the clause and end "
        "your response with exactly: ENFORCEABLE, UNENFORCEABLE, or JURISDICTION-DEPENDENT."
    )


async def run_consensus_swarm():
    swarm = Swarm(
        agents=[USCounsel, EUCounsel, CommonwealthCounsel],
        goal=CLAUSE,
        budget=Budget(max_cost=0.20),
        config=SwarmConfig(
            topology=SwarmTopology.CONSENSUS,
            consensus=ConsensusConfig(
                min_agreement=0.67,  # two of three must agree
                strategy=ConsensusStrategy.MAJORITY,
            ),
        ),
    )
    result = await swarm.run()

    print("Consensus verdict:")
    print(result.content[:300])

    if result.consensus_result:
        cr = result.consensus_result
        print(f"\nConsensus reached: {cr.consensus_reached}")
        print(f"Agreement: {cr.agreement_fraction:.0%}")
        print(f"Winning answer: {cr.winning_answer[:60]}")


asyncio.run(run_consensus_swarm())

---
## 13. Multi-Agent: Reflection

A **producer** writes content. A **critic** scores it and gives feedback. The producer revises. Loops until score threshold or `max_rounds`. Use when quality matters more than speed.

```
Goal ──→ [Writer] ──→ [Editor scores] ──→ score ≥ 0.85? stop
              ↑               │ feedback                ↓ continue
              └───────────────┘  ←─────────── [Writer revises]
```

In [ ]:
from syrin.swarm import ReflectionConfig


class WriterAgent(Agent):
    model = get_model()
    system_prompt = (
        "You write clear, accurate 3-sentence technical explanations. "
        "When given feedback, revise your draft to incorporate it."
    )


class EditorAgent(Agent):
    model = get_model()
    system_prompt = (
        "You review technical explanations for accuracy and clarity. "
        "Give 1-2 specific improvement suggestions. "
        "Always end your response with: 'Score: X.XX' (0.0–1.0)."
    )


async def run_reflection_swarm():
    swarm = Swarm(
        agents=[WriterAgent(), EditorAgent()],
        goal="Explain how vector embeddings enable semantic search in RAG systems",
        budget=Budget(max_cost=0.20),
        config=SwarmConfig(
            topology=SwarmTopology.REFLECTION,
            reflection=ReflectionConfig(
                producer=WriterAgent,
                critic=EditorAgent,
                max_rounds=3,
                stop_when=lambda ro: ro.score >= 0.80,  # early stop on quality
            ),
        ),
    )
    result = await swarm.run()

    print("Final output:")
    print(result.content[:300])

    if result.reflection_result:
        rr = result.reflection_result
        print(f"\nRounds completed: {rr.rounds_completed}")
        for ro in rr.round_outputs:
            print(f"  Round {ro.round_index}: score={ro.score:.2f}")


asyncio.run(run_reflection_swarm())

---
## 14. Multi-Agent: Workflow

Define structured multi-step pipelines: sequential, parallel, conditional, and dynamic fan-out. Output of each step flows into the next via `HandoffContext`.

In [ ]:
from syrin.workflow import Workflow


class PlannerAgent(Agent):
    model = get_model()
    system_prompt = "You create a 3-step research plan. Be concise."


class ResearchAgent(Agent):
    model = get_model()
    system_prompt = "You research a topic and return key findings. Be concise."


class SummaryAgent(Agent):
    model = get_model()
    system_prompt = "You summarise research findings in 2 sentences."


class DetailAgent(Agent):
    model = get_model()
    system_prompt = "You expand research into a detailed 5-sentence report."


async def run_workflow():
    # Sequential workflow — output of each step feeds the next
    wf = (
        Workflow("research-pipeline")
        .step(PlannerAgent, task="Plan research on AI agent frameworks")
        .step(ResearchAgent)  # receives PlannerAgent's output via HandoffContext
        .branch(
            condition=lambda ctx: len(ctx.result.content) > 100,  # if output is substantial
            if_true=SummaryAgent,  # summarise if long
            if_false=DetailAgent,  # expand if short
        )
    )

    swarm = Swarm(
        agents=[],
        goal="Research AI agent frameworks",
        budget=Budget(max_cost=0.30),
        config=SwarmConfig(topology=SwarmTopology.WORKFLOW),
        workflow=wf,
    )
    result = await swarm.run()

    print("Workflow result:")
    print(result.content[:300])


asyncio.run(run_workflow())

In [ ]:
# Parallel fan-out then merge — all agents run simultaneously
class TopicResearcher(Agent):
    model = get_model()
    system_prompt = "You research the assigned topic. Be concise."


class MergeAgent(Agent):
    model = get_model()
    system_prompt = "You synthesise multiple research outputs into a cohesive summary."


async def run_parallel_workflow():
    wf = (
        Workflow("parallel-research")
        .parallel(
            [
                (TopicResearcher, "Research: LLM cost trends 2025"),
                (TopicResearcher, "Research: LLM latency improvements 2025"),
                (TopicResearcher, "Research: LLM context window growth 2025"),
            ]
        )
        .step(MergeAgent)  # receives all three results merged
    )

    result = await wf.run("Comprehensive LLM trends 2025")
    print("Parallel workflow result:")
    print(result.content[:300])


asyncio.run(run_parallel_workflow())

---
## 15. Budget Delegation — `spawn()` and `spawn_many()`

An orchestrator agent can spawn child agents from inside `arun()`, giving each a slice of the shared budget pool. Unspent budget returns to the pool when the child finishes.

In [ ]:
from syrin.response import Response
from syrin.swarm import SpawnSpec, Swarm


class ResearchWorker(Agent):
    model = get_model()
    system_prompt = "You research a topic and return 3 key findings. Be concise."


class FactChecker(Agent):
    model = get_model()
    system_prompt = "You verify research claims and rate accuracy 1-10."


class OrchestratorAgent(Agent):
    model = get_model()
    system_prompt = "You coordinate research and fact-checking sub-agents."

    async def arun(self, input_text: str) -> Response[str]:
        # Single spawn — draws $0.15 from the shared pool
        research = await self.spawn(ResearchWorker, task=input_text, budget=0.15)
        print(f"  Research: cost=${research.cost:.4f}, remaining=${research.budget_remaining:.4f}")

        # Concurrent spawn — both children run in parallel from the pool
        results = await self.spawn_many(
            [
                SpawnSpec(agent=FactChecker, task=research.content, budget=0.08),
                SpawnSpec(
                    agent=FactChecker, task=f"Secondary check: {research.content}", budget=0.08
                ),
            ]
        )
        for r in results:
            print(f"  FactCheck: cost=${r.cost:.4f} | {r.content[:60]}")

        combined = f"Research: {research.content}\n\nVerification: {results[0].content if results else 'n/a'}"
        return Response(content=combined, cost=0.0)


async def run_budget_delegation():
    swarm = Swarm(
        agents=[OrchestratorAgent()],
        goal="Research AI inference optimisation techniques",
        budget=Budget(max_cost=0.50),  # shared pool
    )
    result = await swarm.run()
    print("\nFinal:", result.content[:200])
    if result.budget_report:
        print(f"Total spent: ${result.budget_report.total_spent:.4f}")


asyncio.run(run_budget_delegation())

In [ ]:
# SpawnResult fields
# spawn() returns a SpawnResult with:
#   .content          — the child agent's output text
#   .cost             — actual cost incurred by the child
#   .budget_remaining — pool balance after child completes
#   .stop_reason      — why child terminated (END_TURN, BUDGET, ...)
#   .child_agent_id   — unique ID in "parent::child::uuid" format
print("SpawnResult fields: content, cost, budget_remaining, stop_reason, child_agent_id")

---
## 16. MemoryBus — Cross-Agent Memory

`MemoryBus` lets agents share memory selectively. Publishers write; consumers query. Access is controlled by type filters and lambdas — private entries never cross boundaries.

In [ ]:
from datetime import datetime

from syrin.enums import MemoryType
from syrin.memory.config import MemoryEntry
from syrin.swarm import MemoryBus

# Create a shared MemoryBus — restrict to KNOWLEDGE entries, block private
bus = MemoryBus(
    allow_types=[MemoryType.KNOWLEDGE],
    filter=lambda entry: "private" not in (entry.keywords or []),
)


async def demo_memory_bus():
    # ResearchAgent publishes findings
    finding = MemoryEntry(
        id="finding-001",
        content="LLM inference: speculative decoding reduces latency by 2-3x.",
        type=MemoryType.KNOWLEDGE,
        importance=0.9,
        keywords=["inference", "latency", "speculative-decoding"],
        created_at=datetime.now(),
    )
    stored = await bus.publish(finding, agent_id="researcher")
    print(f"Published: {stored}")

    # Private finding — blocked by filter
    private = MemoryEntry(
        id="priv-001",
        content="Internal cost model details.",
        type=MemoryType.KNOWLEDGE,
        importance=0.5,
        keywords=["private", "costs"],
        created_at=datetime.now(),
    )
    stored_priv = await bus.publish(private, agent_id="researcher")
    print(f"Private blocked: {stored_priv}")

    # AnalysisAgent queries — gets relevant findings, not private ones
    results = await bus.read(query="speculative decoding latency", agent_id="analyst")
    print(f"\nAnalyst found {len(results)} finding(s):")
    for entry in results:
        print(f"  [{entry.type}] {entry.content[:80]}")


asyncio.run(demo_memory_bus())

In [ ]:
# Use MemoryBus in a Swarm — inject at construction, pass instances not classes
shared_bus = MemoryBus(allow_types=[MemoryType.KNOWLEDGE])


class PublisherAgent(Agent):
    model = get_model()
    system_prompt = "You research and publish findings."

    def __init__(self, memory_bus: MemoryBus, **kwargs):
        super().__init__(**kwargs)
        self._bus = memory_bus


class ConsumerAgent(Agent):
    model = get_model()
    system_prompt = "You read from shared memory and synthesise findings."

    def __init__(self, memory_bus: MemoryBus, **kwargs):
        super().__init__(**kwargs)
        self._bus = memory_bus


# Pass instances (not classes) so bus injection works
# swarm = Swarm(
#     agents=[PublisherAgent(shared_bus), ConsumerAgent(shared_bus)],
#     goal="Research and synthesise AI trends",
#     config=SwarmConfig(topology=SwarmTopology.ORCHESTRATOR),
# )
print("MemoryBus injection pattern: pass agent instances with shared bus in __init__")

---
## 17. A2A Messaging — Agent-to-Agent

Typed, async message passing between agents. Three channels: **direct** (one-to-one), **broadcast** (all agents), **topic** (pub/sub).

In [ ]:
from dataclasses import dataclass

from syrin.swarm import A2AConfig, A2ARouter


@dataclass
class TaskAssignment:
    task_id: str
    topic: str
    priority: int = 1


@dataclass
class TaskResult:
    task_id: str
    summary: str
    confidence: float


@dataclass
class PhaseUpdate:
    phase: str
    status: str


async def demo_a2a():
    # Set up router with audit logging
    router = A2ARouter(config=A2AConfig(audit_all=True))
    for agent_id in ("orchestrator", "researcher", "analyst", "reviewer"):
        router.register_agent(agent_id)

    # ── Direct: orchestrator → researcher ────────────────────────────────────
    print("=== Direct Messaging ===")
    await router.send(
        from_agent="orchestrator",
        to_agent="researcher",
        message=TaskAssignment(task_id="t-001", topic="LLM inference optimisation", priority=2),
    )
    envelope = await router.receive(agent_id="researcher", timeout=1.0)
    if envelope:
        task = envelope.payload
        print(f"Researcher received: {task.topic} (priority={task.priority})")

        # Researcher replies
        await router.send(
            from_agent="researcher",
            to_agent="orchestrator",
            message=TaskResult(
                task_id=task.task_id, summary="Speculative decoding = 2-3x speedup", confidence=0.91
            ),
        )

    reply = await router.receive(agent_id="orchestrator", timeout=1.0)
    if reply:
        result = reply.payload
        print(f"Orchestrator received: {result.summary} (confidence={result.confidence:.0%})")

    # ── Topic: subscribe + fan-out ────────────────────────────────────────────
    print("\n=== Topic (Pub/Sub) ===")
    router.subscribe("analyst", topic="pipeline")
    router.subscribe("reviewer", topic="pipeline")

    await router.send_topic(
        from_agent="orchestrator",
        topic="pipeline",
        message=PhaseUpdate(phase="research", status="complete"),
    )

    for agent_id in ("analyst", "reviewer"):
        env = await router.receive(agent_id=agent_id, timeout=0.5)
        if env:
            update = env.payload
            print(f"  {agent_id}: phase='{update.phase}' status='{update.status}'")

    # ── Audit log ─────────────────────────────────────────────────────────────
    print("\n=== Audit Log ===")
    for entry in router.audit_log():
        print(f"  {entry.from_agent:<14} → {entry.to_agent:<12} [{entry.message_type}]")


asyncio.run(demo_a2a())

---
## 18. Runtime Budget Control

After agents are running, a supervisor can adjust budgets without stopping anything. `topup_budget()` adds to an existing allocation. `reallocate_budget()` replaces it entirely. Both are atomic and audit-logged.

In [ ]:
from syrin.budget._pool import BudgetPool
from syrin.enums import AgentRole, AgentStatus
from syrin.swarm._authority import SwarmAuthorityGuard
from syrin.swarm._control import AgentStateSnapshot, SwarmController


def make_snapshot(agent_id: str, supervisor: str | None = None) -> AgentStateSnapshot:
    return AgentStateSnapshot(
        agent_id=agent_id,
        status=AgentStatus.RUNNING,
        role=AgentRole.WORKER,
        last_output_summary="",
        cost_spent=0.0,
        task="working",
        context_override=None,
        supervisor_id=supervisor,
    )


async def demo_runtime_budget():
    # $10 pool — CTO gets $2, marketing gets $4
    pool = BudgetPool(total=10.00)
    await pool.allocate("cto", 2.00)
    await pool.allocate("marketing", 4.00)

    guard = SwarmAuthorityGuard(
        roles={
            "ceo": AgentRole.ORCHESTRATOR,
            "cto": AgentRole.WORKER,
            "marketing": AgentRole.WORKER,
        },
        teams={"ceo": ["cto", "marketing"]},
    )
    ctrl = SwarmController(
        actor_id="ceo",
        guard=guard,
        state_registry={
            "cto": make_snapshot("cto", "ceo"),
            "marketing": make_snapshot("marketing", "ceo"),
        },
        task_registry={},
        budget_pool=pool,
    )

    snap = pool.snapshot()
    print("Before rebalance:")
    for k, v in snap.items():
        print(f"  {k}: allocated=${v['allocated']:.2f} spent=${v['spent']:.2f}")
    print(f"  Pool remaining: ${pool.remaining:.2f}")

    # CEO tops up CTO mid-run (draws $1.50 from pool)
    await ctrl.topup_budget("cto", 1.50)

    # CEO reallocates marketing down (returns $2 to pool)
    await ctrl.reallocate_budget("marketing", 2.00)

    snap = pool.snapshot()
    print("\nAfter rebalance:")
    for k, v in snap.items():
        print(f"  {k}: allocated=${v['allocated']:.2f}")
    print(f"  Pool remaining: ${pool.remaining:.2f}")

    # Audit trail
    print("\nAudit log:")
    for entry in ctrl._guard.audit_log():
        print(f"  {entry.action:<24} {entry.actor_id} → {entry.target_id}")


asyncio.run(demo_runtime_budget())

---
## 19. AgentRouter — Dynamic Routing

The LLM decides which agents to call, how many, and in what order. Different from a fixed workflow — the routing decision adapts per input at runtime.

In [ ]:
from syrin.agent.agent_router import AgentRouter


class MarketDataAgent(Agent):
    name = "market_data"
    model = get_model()
    system_prompt = "You find market data, statistics, and growth rates. Be concise."


class CompetitorAgent(Agent):
    name = "competitor_analysis"
    model = get_model()
    system_prompt = "You analyse competitors: positioning, pricing, market share. Be concise."


class ReportWriterAgent(Agent):
    name = "report_writer"
    model = get_model()
    system_prompt = "You write investment briefs from research. Be concise."


# Router — LLM plans which agents to use
router = AgentRouter(
    agents=[MarketDataAgent, CompetitorAgent, ReportWriterAgent],
    model=get_model(),
    budget=Budget(max_cost=0.30),
)

# router.visualize()  # shows agent pool

# Parallel mode — LLM-selected agents run simultaneously
result = router.run(
    task="Research AI coding assistants market and write an investment brief",
    mode="parallel",  # or "sequential" — each agent sees prior output
)

print("Router result:")
print(result.content[:300])
print(f"\nCost: ${result.cost:.6f}")

---
## 20. Hierarchical Swarms

Model org-chart relationships with `Agent.team`. Pass only the CEO — Syrin expands the full hierarchy automatically and sets `supervisor_id` on each agent.

```
CEO
 ├── CTO
 │    └── EngineeringLead
 │         ├── BackendEngineer
 │         └── FrontendEngineer
 └── CMO
      └── MarketingLead
           └── ContentWriter
```

In [ ]:
from typing import ClassVar


# ── Leaf workers ──────────────────────────────────────────────────────────────
class BackendEngineer(Agent):
    model = get_model()
    system_prompt = (
        "You describe backend work needed for a feature (API, data model, risk). Be concise."
    )


class FrontendEngineer(Agent):
    model = get_model()
    system_prompt = "You describe frontend work needed for a feature (components, UX). Be concise."


class ContentWriter(Agent):
    model = get_model()
    system_prompt = "You describe content deliverables for a feature (copy, docs). Be concise."


# ── Middle management ─────────────────────────────────────────────────────────
class EngineeringLead(Agent):
    model = get_model()
    system_prompt = "You create the engineering plan for a feature. Be concise."
    team: ClassVar[list] = [BackendEngineer, FrontendEngineer]


class MarketingLead(Agent):
    model = get_model()
    system_prompt = "You create the go-to-market plan for a feature. Be concise."
    team: ClassVar[list] = [ContentWriter]


# ── C-suite ────────────────────────────────────────────────────────────────────
class CTO(Agent):
    model = get_model()
    system_prompt = "You set the technical strategy for a feature. Be concise."
    team: ClassVar[list] = [EngineeringLead]


class CMO(Agent):
    model = get_model()
    system_prompt = "You set the marketing strategy for a feature. Be concise."
    team: ClassVar[list] = [MarketingLead]


class CEO(Agent):
    model = get_model()
    system_prompt = "You set the strategic direction for a feature. Be concise."
    team: ClassVar[list] = [CTO, CMO]  # Swarm expands this recursively


async def run_hierarchical_swarm():
    swarm = Swarm(
        agents=[CEO()],  # pass only the CEO — full tree is expanded automatically
        goal="Launch an AI-powered code review feature for enterprise engineering teams.",
        budget=Budget(max_cost=0.50),
        config=SwarmConfig(topology=SwarmTopology.PARALLEL),
    )

    print(f"Agent pool after expansion: {swarm.agent_count} agents")

    result = await swarm.run()
    print("\nCombined output from all levels:")
    print(result.content[:400])

    if result.budget_report:
        print(f"\nTotal spent: ${result.budget_report.total_spent:.4f}")


asyncio.run(run_hierarchical_swarm())

In [ ]:
# Inspect supervisor chain — each agent knows its direct supervisor
async def inspect_hierarchy():
    swarm = Swarm(agents=[CEO()], goal="inspect")
    print(f"{'Agent':<25} {'Supervisor'}")
    print("-" * 50)
    for agent in swarm._agents:
        supervisor = getattr(agent, "_supervisor_id", None)
        print(f"{type(agent).__name__:<25} {supervisor or '(root)'}")


asyncio.run(inspect_hierarchy())

---
## 21. AgentRegistry

Discover and control agents by name or capability tag. Created automatically by `Swarm`. Also usable standalone.

In [ ]:
from syrin.swarm._registry import AgentRegistry

registry = AgentRegistry(heartbeat_interval=5.0)

# Register agents with capability tags
registry.register(
    Agent(model=get_model(), system_prompt="Finance analyst."), tags=["finance", "analysis"]
)
registry.register(
    Agent(model=get_model(), system_prompt="Research specialist."), tags=["research", "analysis"]
)

# Discover agents
all_agents = registry.all()
print(f"All agents: {len(all_agents)}")

analysis_agents = registry.find(tag="analysis")
print(f"Analysis agents: {len(analysis_agents)}")

# List with status
for entry in registry.list_agents():
    print(f"  {entry.agent_id}: {entry.status}")

---
## 22. Serving — HTTP & Playground

`agent.serve()` exposes your agent over HTTP. `enable_playground=True` adds a browser UI. Run in a terminal — `serve()` blocks.

In [ ]:
# HTTP endpoints after agent.serve(port=8000):
#   POST /chat        — send {"message": "..."}, get response
#   POST /stream      — streaming response
#   GET  /playground  — browser chat UI (when enable_playground=True)
#   GET  /graph       — Mermaid diagram (for Swarm/Workflow)

agent = Agent(
    model=get_model(),
    system_prompt="You are a helpful assistant.",
    budget=Budget(max_cost=0.50, exceed_policy=ExceedPolicy.WARN),
)

# Run this in a terminal:
#   pip install syrin[serve]
#   python -c "from syrin import Agent, Model; a=Agent(model=Model.mock()); a.serve(port=8000, enable_playground=True)"
#   # Visit http://localhost:8000/playground

# Swarm can also be served:
# swarm = Swarm(agents=[ResearchAgent, WriterAgent], goal="...")
# swarm.serve(port=8001)  # GET /graph returns Mermaid diagram

print("To serve: agent.serve(port=8000, enable_playground=True)")
print("Endpoints: POST /chat, POST /stream, GET /playground, GET /graph")

---
## 23. Real-World Example: Research Pipeline

A complete end-to-end example combining everything: a 3-level orchestrated swarm that plans, researches in parallel, fact-checks, and synthesises — with budget delegation and cross-agent memory.

**Use case:** AI-powered market research desk. CEO delegates to analysts; analysts research in parallel; fact-checker verifies; CEO synthesises final report.

In [ ]:
# ── Specialist researchers ────────────────────────────────────────────────────
class MarketSizeResearcher(Agent):
    model = get_model()
    system_prompt = (
        "You research market size, CAGR, and revenue forecasts. "
        "Return 3 specific data points with sources. Be concise."
    )


class CompetitorResearcher(Agent):
    model = get_model()
    system_prompt = (
        "You research key competitors: market share, moat, pricing. "
        "Name top 3 players with a specific differentiator each. Be concise."
    )


class RiskResearcher(Agent):
    model = get_model()
    system_prompt = (
        "You identify market risks: regulatory, technical, macro. "
        "List top 3 risks with impact rating. Be concise."
    )


class FactCheckerAgent(Agent):
    model = get_model()
    system_prompt = (
        "You verify research claims. Flag any unsupported assertions. "
        "Rate overall accuracy 1-10. Be concise."
    )


# ── CEO orchestrator — spawns all children with budget slices ─────────────────
class CEOOrchestrator(Agent):
    model = get_model()
    system_prompt = (
        "You are a research CEO. You coordinate specialist researchers and "
        "synthesise their outputs into a structured investment brief: "
        "Market Overview, Competitive Landscape, Risk Assessment, Recommendation."
    )

    async def arun(self, topic: str) -> Response[str]:
        print(f"  [CEO] Launching parallel research on: {topic[:60]}")

        # Spawn 3 researchers in parallel — each gets $0.10 from pool
        research_results = await self.spawn_many(
            [
                SpawnSpec(agent=MarketSizeResearcher, task=topic, budget=0.10),
                SpawnSpec(agent=CompetitorResearcher, task=topic, budget=0.10),
                SpawnSpec(agent=RiskResearcher, task=topic, budget=0.10),
            ]
        )

        for r in research_results:
            label = r.child_agent_id.split("::")[1] if "::" in r.child_agent_id else "researcher"
            print(f"  [CEO] {label}: cost=${r.cost:.4f} | {r.content[:60]}...")

        # Combine all research
        combined_research = "\n\n".join(
            [
                f"Market Data:\n{research_results[0].content}",
                f"Competition:\n{research_results[1].content}",
                f"Risks:\n{research_results[2].content}",
            ]
        )

        # Spawn fact-checker on combined research
        fact_check = await self.spawn(FactCheckerAgent, task=combined_research, budget=0.08)
        print(f"  [CEO] Fact-check: accuracy={fact_check.content[:40]}...")

        # CEO synthesises final brief
        brief = (
            f"=== INVESTMENT BRIEF: {topic.upper()} ===\n\n"
            f"{combined_research}\n\n"
            f"Verification: {fact_check.content}\n\n"
            f"[Synthesised by CEO Orchestrator]"
        )
        total_child_cost = sum(r.cost for r in research_results) + fact_check.cost
        return Response(content=brief, cost=total_child_cost)


async def run_research_pipeline():
    print("Starting Research Pipeline...\n")

    swarm = Swarm(
        agents=[CEOOrchestrator()],
        goal="AI agent frameworks market — investment analysis 2025",
        budget=Budget(max_cost=1.00),  # shared pool for all spawned agents
    )

    result = await swarm.run()

    print("\n" + "=" * 60)
    print("FINAL REPORT (first 500 chars):")
    print(result.content[:500])

    if result.budget_report:
        print(f"\nTotal cost: ${result.budget_report.total_spent:.4f}")


asyncio.run(run_research_pipeline())

---
## What's Next?

| Resource | Description |
|----------|-------------|
| `examples/07_multi_agent/` | 18 runnable multi-agent examples with real LLM calls |
| `examples/03_budget/` | Budget estimation, guardrails, forecasting, anomaly detection |
| `docs/multi-agent/` | Deep-dive docs: swarm, workflow, authority, MemoryBus, A2A |
| `docs/api-reference.md` | Complete API reference for all classes |
| `plan/v0.12.0.md` | What's coming: Swarm State Layer, Goal Drift, Agent Harness |

### Key Patterns

**Virtual Office** — `CEO.team = [CTO, CMO]` → hierarchical swarm with budget control per level.

**Research Swarm** — `spawn_many([...])` with parallel researchers → fact-checker → synthesiser.

**Quality Loop** — `REFLECTION` topology with `stop_when=lambda ro: ro.score >= 0.85`.

**Safety Committee** — `CONSENSUS` with `min_agreement=0.67` across 3+ independent agents.

**Dynamic Pipeline** — `Workflow.dynamic(lambda ctx: [(Agent, task, budget) for task in ctx.data.tasks])`.